# Experiment 7 — Store Weight Profile

**Hypothesis:** Do some outlets specialise in heavier items (bulk staples)
vs lighter items (snacks, beverages)? If so, the fraction of lightweight items
at each outlet could characterise store format beyond what `Outlet_Type` captures.

**What the data showed:**

| Metric | Value |
|---|---|
| Weight–MRP correlation | 0.026 (near zero — heavy ≠ expensive) |
| Weight–sales correlation | 0.013 (near zero) |
| Store mean weight range | 12.65–13.01 across all 10 outlets |

All 10 outlets stock essentially the same 1,559 items — there is no weight-based
specialisation. The hypothesis was wrong.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

SEEDS = [42, 7, 13, 99, 2024, 123, 456, 17, 31, 37,
         53, 71, 83, 97, 101, 127, 131, 137, 149, 163]
KF = 5

# ── 1. Load ───────────────────────────────────────────────────────────────────
train = pd.read_csv('train_v9rqX0R.csv')
test  = pd.read_csv('test_AbJTz2l.csv')
print(f"Train: {train.shape} | Test: {test.shape}")

# ── 2. Clean ──────────────────────────────────────────────────────────────────
def clean(df, rw=None, rv=None):
    df = df.copy()
    df['Item_Fat_Content'] = df['Item_Fat_Content'].replace(
        {'LF':'Low Fat','low fat':'Low Fat','reg':'Regular'})
    df['Item_Category'] = df['Item_Identifier'].str[:2]
    df.loc[df['Item_Category']=='NC','Item_Fat_Content'] = 'Non-Consumable'
    if rw is None: rw = df.groupby('Item_Identifier')['Item_Weight'].mean()
    df['Item_Weight'] = df.apply(
        lambda r: rw.get(r['Item_Identifier'], np.nan)
        if pd.isna(r['Item_Weight']) else r['Item_Weight'], axis=1)
    df['Item_Weight'].fillna(df['Item_Weight'].median(), inplace=True)
    df['Outlet_Size'] = df.groupby('Outlet_Type')['Outlet_Size'].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Medium'))
    df['Outlet_Age']  = 2013 - df['Outlet_Establishment_Year']
    df['MRP_Cluster'] = pd.cut(df['Item_MRP'], bins=[0,70,130,200,270],
                                labels=[0,1,2,3]).astype(int)
    return df, rw, rv

tc, rw, rv = clean(train)
sc, _,  _  = clean(test, rw, rv)

om = tc.groupby('Outlet_Identifier')['Item_MRP'].mean()
tc['Outlet_MRP_Mean'] = tc['Outlet_Identifier'].map(om)
sc['Outlet_MRP_Mean'] = sc['Outlet_Identifier'].map(om)

for col in ['Item_Fat_Content','Outlet_Size','Outlet_Location_Type',
            'Outlet_Type','Item_Category']:
    le = LabelEncoder().fit(pd.concat([tc[col], sc[col]]).astype(str))
    tc[col] = le.transform(tc[col].astype(str))
    sc[col] = le.transform(sc[col].astype(str))

# ── 3. Interactions ───────────────────────────────────────────────────────────
tc['Age_x_Size']           = tc['Outlet_Age'] * tc['Outlet_Size']
sc['Age_x_Size']           = sc['Outlet_Age'] * sc['Outlet_Size']
tc['Cluster_x_Size_x_Cat'] = tc['MRP_Cluster'] * tc['Outlet_Size'] * tc['Item_Category']
sc['Cluster_x_Size_x_Cat'] = sc['MRP_Cluster'] * sc['Outlet_Size'] * sc['Item_Category']

# ── 4. Store features ─────────────────────────────────────────────────────────
spm = (train.groupby('Outlet_Identifier')['Item_Outlet_Sales'].sum() /
       train.groupby('Outlet_Identifier')['Item_MRP'].sum())
n_items = train.groupby('Outlet_Identifier')['Item_Identifier'].nunique()

# Store weight profile: % of items weighing < 8kg (light items)
# Light items = snacks, beverages, small packaged goods
# Computed from train only (using known weights)
store_pct_light = train.groupby('Outlet_Identifier')['Item_Weight'].apply(
    lambda x: (x.dropna() < 8).mean())

for df, src in [(tc, train), (sc, test)]:
    df['store_n_items']    = src['Outlet_Identifier'].map(n_items)
    df['selling_price_est']= src['Item_MRP'].values * src['Outlet_Identifier'].map(spm)
    df['store_pct_light']  = src['Outlet_Identifier'].map(store_pct_light)

# ── 5. Visibility ─────────────────────────────────────────────────────────────
item_vis_mean   = train[train['Item_Visibility']>0].groupby(
    'Item_Identifier')['Item_Visibility'].mean()
global_vis_mean = train[train['Item_Visibility']>0]['Item_Visibility'].mean()

for df, src in [(tc, train), (sc, test)]:
    vis = src['Item_Visibility'].values.astype(float).copy()
    zero_mask = (vis == 0)
    vis[zero_mask] = src['Item_Identifier'].map(
        item_vis_mean).fillna(global_vis_mean).values[zero_mask]
    df['vis_filled'] = vis

q5_bins = np.quantile(tc['vis_filled'].values, [0, .2, .4, .6, .8, 1.0])
for df in [tc, sc]:
    df['vis_bucket_q5'] = np.searchsorted(
        q5_bins[1:], df['vis_filled'].values, side='right').clip(0, 4)
    df['vis_x_outsize'] = df['vis_filled'] * df['Outlet_Size']

# ── 6. Features ───────────────────────────────────────────────────────────────
feats = [
    'Item_MRP',             'MRP_Cluster',          'Outlet_MRP_Mean',
    'Outlet_Type',          'Outlet_Age',            'Outlet_Size',
    'Outlet_Location_Type', 'Item_Category',         'Item_Fat_Content',
    'Age_x_Size',           'Cluster_x_Size_x_Cat',
    'store_n_items',        'selling_price_est',
    'vis_filled',           'vis_bucket_q5',         'vis_x_outsize',
    'store_pct_light',
]

for df in [tc, sc]:
    for f in feats: df[f] = df[f].fillna(0)

X      = tc[feats].values.astype(float)
y      = train['Item_Outlet_Sales'].values
X_test = sc[feats].values.astype(float)
print(f"Features: {len(feats)}")

# ── 7. RF 20 seeds, leaf=32 ───────────────────────────────────────────────────
print(f"\n=== RF (leaf=32, {len(SEEDS)} seeds) ===")
all_oofs  = []
all_tests = []

for seed in SEEDS:
    kf  = KFold(n_splits=KF, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))
    tp  = np.zeros(len(X_test))
    for tr_i, val_i in kf.split(X):
        m = RandomForestRegressor(
            n_estimators=500, min_samples_leaf=32,
            max_features=0.5, n_jobs=-1, random_state=seed)
        m.fit(X[tr_i], y[tr_i])
        oof[val_i] = m.predict(X[val_i])
        tp += m.predict(X_test) / KF

    rmse = np.sqrt(mean_squared_error(y, oof))
    print(f"  seed={seed:4d}  OOF={rmse:.2f}", flush=True)
    all_oofs.append(oof); all_tests.append(tp)

    if len(all_oofs) % 5 == 0:
        tp_now  = np.mean(all_tests, axis=0)
        oof_now = np.mean(all_oofs,  axis=0)
        sub = test[['Item_Identifier','Outlet_Identifier']].copy()
        sub['Item_Outlet_Sales'] = tp_now.clip(min=0)
        sub.to_csv(f'submission/predictions7_{len(all_oofs)}seeds.csv', index=False)
        print(f"  --> Checkpoint {len(all_oofs)} seeds: "
              f"OOF={np.sqrt(mean_squared_error(y,oof_now)):.2f}  "
              f"std={tp_now.std():.0f}")

# ── 8. Save ───────────────────────────────────────────────────────────────────
oof_final  = np.mean(all_oofs,  axis=0)
test_final = np.mean(all_tests, axis=0)
print(f"\nFinal {len(SEEDS)}-seed OOF: {np.sqrt(mean_squared_error(y,oof_final)):.2f}  "
      f"std={test_final.std():.0f}")

sub = test[['Item_Identifier','Outlet_Identifier']].copy()
sub['Item_Outlet_Sales'] = test_final.clip(min=0)
sub.to_csv('submission/predictions7.csv', index=False)

print(f"  mean={test_final.mean():.0f}  std={test_final.std():.0f}  "
      f"min={test_final.clip(min=0).min():.0f}")

Train: (8523, 12) | Test: (5681, 11)
Features: 17

=== RF (leaf=32, 20 seeds) ===
  seed=  42  OOF=1077.12
  seed=   7  OOF=1077.10
  seed=  13  OOF=1078.98
  seed=  99  OOF=1077.78
  seed=2024  OOF=1078.11
  --> Checkpoint 5 seeds: OOF=1076.19  std=1338
  seed= 123  OOF=1078.72
  seed= 456  OOF=1077.08
  seed=  17  OOF=1076.39
  seed=  31  OOF=1075.81
  seed=  37  OOF=1078.42
  --> Checkpoint 10 seeds: OOF=1075.76  std=1338
  seed=  53  OOF=1079.36
  seed=  71  OOF=1080.08
  seed=  83  OOF=1078.59
  seed=  97  OOF=1077.00
  seed= 101  OOF=1076.43
  --> Checkpoint 15 seeds: OOF=1075.90  std=1338
  seed= 127  OOF=1075.26
  seed= 131  OOF=1077.50
  seed= 137  OOF=1078.50
  seed= 149  OOF=1078.39
  seed= 163  OOF=1077.53
  --> Checkpoint 20 seeds: OOF=1075.77  std=1338

Final 20-seed OOF: 1075.77  std=1338
  mean=2185  std=1338  min=97
